In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PidMLP import PidMLP
from stable_baselines3 import PPO
from environment import Environment

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
action_space_n = 50
action_bins = np.linspace(-2, 2, action_space_n)  # Define bins for discretization

In [8]:
# Load data
df = pd.read_csv('pid_controller_data.csv')

# Seperate features and labels
features = df[['target_lataccel', 'current_lataccel', 'vEgo', 'aEgo', 'roll', 'error', 'prev_error', 'integral', 'derivative', 'prev_derivative', "prev_action"]].values
labels = df['steerCommand'].values

# Discretize labels and one-hot encode
bin_indices = np.digitize(labels, action_bins, right=True) - 1
bin_indices = np.clip(bin_indices, 0, action_space_n - 1)

# Convert to tensors
features_tensor = torch.tensor(features, dtype=torch.float32, device=device)
labels_tensor = torch.tensor(bin_indices, dtype=torch.long, device=device)

# Create a TensorDataset
dataset = TensorDataset(features_tensor, labels_tensor)

# Create a DataLoader
dataloader = DataLoader(dataset, batch_size=8000, shuffle=True,)

In [10]:
len(dataloader)

1249

In [ ]:
env = Environment()

# Instantiate model
student = PPO("MlpPolicy", env, verbose=1)
model = student.policy.to(device)

In [11]:


# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
schedular = optim.lr_scheduler.StepLR(optimizer, 5, 0.5)

# Training loop
epochs = 15
losses = []
for epoch in range(epochs):
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        
        # Calculate loss
        loss = criterion(outputs, targets)
        total_loss += loss.item()
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += targets.size(0)
        correct_predictions += (predicted == targets).sum().item()

    schedular.step()

    # Compute average loss
    average_loss = total_loss / len(dataloader)
    losses.append(average_loss)

    # Compute accuracy
    accuracy = (correct_predictions / total_predictions) * 100
    
    # Print loss and accuracy
    print(f'End of Epoch {epoch+1}, Loss: {average_loss:.4f}, Accuracy: {accuracy:.2f}%')

End of Epoch 1, Loss: 0.6710, Accuracy: 75.05%
End of Epoch 2, Loss: 0.2519, Accuracy: 90.57%
End of Epoch 3, Loss: 0.1929, Accuracy: 92.86%
End of Epoch 4, Loss: 0.1583, Accuracy: 94.18%
End of Epoch 5, Loss: 0.1365, Accuracy: 94.97%
End of Epoch 6, Loss: 0.1123, Accuracy: 95.88%
End of Epoch 7, Loss: 0.1046, Accuracy: 96.15%
End of Epoch 8, Loss: 0.0974, Accuracy: 96.41%
End of Epoch 9, Loss: 0.0927, Accuracy: 96.57%
End of Epoch 10, Loss: 0.0881, Accuracy: 96.74%
End of Epoch 11, Loss: 0.0766, Accuracy: 97.17%
End of Epoch 12, Loss: 0.0743, Accuracy: 97.25%
End of Epoch 13, Loss: 0.0729, Accuracy: 97.31%
End of Epoch 14, Loss: 0.0714, Accuracy: 97.36%
End of Epoch 15, Loss: 0.0690, Accuracy: 97.44%


In [ ]:
model = torch.load('./models/mlp_pid.pth')

In [12]:
torch.save(model, './models/mlp_pid.pth')

In [13]:
model.eval()
action = model(torch.tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device=device, dtype=torch.float32))
print(action)

tensor([-30.0901, -31.0566, -34.4764, -30.0724, -31.3424, -32.8741, -30.1881,
        -28.3736, -28.4353, -27.6183, -24.8627, -25.9261, -22.4459, -23.1588,
        -25.2081, -35.7956, -51.5369, -35.9329, -27.7663, -53.5872, -74.4145,
        -45.0358, -24.1968, -20.0683,  -6.4733,   3.7113,  10.8117,  18.5214,
        -12.9076, -39.7607, -43.4471, -38.8956, -38.8247, -30.8302, -26.6415,
        -23.7837, -22.2876, -29.3385, -33.4944, -42.2610, -50.9144, -55.0635,
        -63.2005, -66.8591, -70.7690, -71.8856, -74.5556, -78.3804, -73.7641,
        -15.0267], device='cuda:0', grad_fn=<AddBackward0>)
